# Setup

## Configure Session
On the right, there is Session Options:
- Accelerator: GPU T4 x2
- Language: Python
- Persistence: No persistence
- Environment: Pin to original environment
- Internet: Internet off


## Add Inputs
Add the following inputs on the right
- https://www.kaggle.com/datasets/st3v3d/2nd-place-byu-challenge-packages
- https://www.kaggle.com/datasets/st3v3d/2nd-place-byu-challenge-checkpoints
- byu-locating-bacterial-flagellar-motors-2025


In [1]:
import os
os.listdir('/kaggle/input')

['byu-locating-bacterial-flagellar-motors-2025',
 '2nd-place-byu-challenge-checkpoints',
 '2nd-place-byu-challenge-packages']

## Install packages

In [2]:
!pip install --no-index --find-links /kaggle/input/2nd-place-byu-challenge-packages/ /kaggle/input/2nd-place-byu-challenge-packages/*.whl

Looking in links: /kaggle/input/2nd-place-byu-challenge-packages/
Processing /kaggle/input/2nd-place-byu-challenge-packages/acvl_utils-0.2.5-py3-none-any.whl
Processing /kaggle/input/2nd-place-byu-challenge-packages/argparse-1.4.0-py2.py3-none-any.whl
Processing /kaggle/input/2nd-place-byu-challenge-packages/batchgenerators-0.25.1-py3-none-any.whl
Processing /kaggle/input/2nd-place-byu-challenge-packages/batchgeneratorsv2-0.2.3-py3-none-any.whl
Processing /kaggle/input/2nd-place-byu-challenge-packages/blosc2-3.3.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/input/2nd-place-byu-challenge-packages/build-1.2.2.post1-py3-none-any.whl
Processing /kaggle/input/2nd-place-byu-challenge-packages/certifi-2025.4.26-py3-none-any.whl
Processing /kaggle/input/2nd-place-byu-challenge-packages/challenge2025_kaggle_byu_flagellarmotors-0.0.1-py3-none-any.whl
Processing /kaggle/input/2nd-place-byu-challenge-packages/charset_normalizer-3.4.2-cp311-cp311-manylinux_2_17_x86

## Create Inference Script

In [3]:
# write the entire inference script to a file
script = r"""
import os
import argparse
import ast
from concurrent.futures import ThreadPoolExecutor

import torch
import numpy as np
import pandas as pd
from batchgenerators.utilities.file_and_folder_operations import subdirs, join
from torch.nn.functional import interpolate

from nnunetv2.dataset_conversion.kaggle_byu.official_data_to_nnunet import convert_coordinates, load_jpgs
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
from nnunetv2.utilities.helpers import empty_cache


@torch.inference_mode()
def resize_image(image: np.ndarray, edge_length: int, device: torch.device) -> torch.Tensor:
    zoom = edge_length / max(image.shape)
    new_shape = [round(s * zoom) for s in image.shape]
    t = torch.from_numpy(image).to(device).float()
    t = interpolate(t[None, None], new_shape, mode='area')[0, 0]
    t = torch.clip(torch.round(t), 0, 255).byte()
    empty_cache(device)
    return t


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument('--input-dir',
                   default="/kaggle/input/byu-locating-bacterial-flagellar-motors-2025/test")
    p.add_argument('--output-file',
                   default="/kaggle/working/submission.csv",
                   help="output-file path")
    p.add_argument('--ckpt-dir')
    p.add_argument(
        '--fold',
        type=ast.literal_eval,
        help="tuple of fold identifiers, e.g. ('all',) or (0,1,2)"
    )
    p.add_argument('--threshold', type=float)
    p.add_argument('--min-dist', type=int, default=13)
    p.add_argument('--edge', type=int, default=512)
    p.add_argument('--gpu-id', type=int, default=0,
                   help="This process's GPU index (0 to num_gpus-1)")
    p.add_argument('--num-gpus', type=int, default=1,
                   help="Total number of GPUs being used")
    return p.parse_args()

def main():
    args = parse_args()
    DEVICE = torch.device(f"cuda:{args.gpu_id}")

    # ensure unique filename per GPU
    out_path = args.output_file

    all_tomos = sorted(subdirs(args.input_dir, join=False))
    tomos = all_tomos[args.gpu_id::args.num_gpus]

    pred = nnUNetPredictor(
        tile_step_size=0.5,
        use_gaussian=True,
        use_mirroring=True,
        perform_everything_on_device=True,
        device=DEVICE,
        verbose=False,
        verbose_preprocessing=False,
        allow_tqdm=True
    )
    pred.initialize_from_trained_model_folder(args.ckpt_dir, args.fold)
    pred.label_manager._all_labels = [0]

    results = []
    with ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(load_jpgs, join(args.input_dir, tomos[0]))
        for i, tomo in enumerate(tomos):
            img_np = future.result()
            if i + 1 < len(tomos):
                future = executor.submit(load_jpgs, join(args.input_dir, tomos[i+1]))

            orig_shape = img_np.shape
            img = resize_image(img_np, args.edge, DEVICE).float()
            img = (img - img.mean()) / img.std()

            out = pred.predict_logits_from_preprocessed_data(img[None], out_device=DEVICE).float()[None]
            out = torch.sigmoid(out)[0, 0]
            coords = torch.argwhere((out == torch.max(out)) & (out > args.threshold))
            ps = [out[tuple(c)].item() for c in coords]

            if len(ps) == 0:
                results.append({'tomo_id': tomo,
                                'Motor axis 0': -1,
                                'Motor axis 1': -1,
                                'Motor axis 2': -1})
            else:
                # all motors equally likely, pick first
                best = coords[0].tolist()
                xyz = convert_coordinates([best], img.shape, orig_shape)[0]
                results.append({'tomo_id': tomo,
                                'Motor axis 0': xyz[0],
                                'Motor axis 1': xyz[1],
                                'Motor axis 2': xyz[2]})

            # free up memory
            del img, out, coords, ps, img_np
            empty_cache(DEVICE)

    # write out clean CSV
    df = pd.DataFrame(results, columns=['tomo_id','Motor axis 0','Motor axis 1','Motor axis 2'])
    df.to_csv(out_path, index=False)
    print(f"Saved predictions to {out_path}")

if __name__ == "__main__":
    main()
"""
script_path = '/kaggle/working/inference.py'
with open(script_path, 'w') as f:
    f.write(script)
print(f"Saved {script_path}")


Saved /kaggle/working/inference.py


# Execute

## Spawn two processes with each one GPU attached

In [4]:
import os, sys, subprocess

os.environ['nnUNet_compile'] = 'True'
os.environ['torch.backends.cudnn.benchmark'] = 'True'

base = [
    sys.executable,
    "/kaggle/working/inference.py",
    "--num-gpus", "2",
    "--input-dir", "/kaggle/input/byu-locating-bacterial-flagellar-motors-2025/test",
    "--ckpt-dir", "/kaggle/input/2nd-place-byu-challenge-checkpoints/MotorRegressionTrainer_BCEtopK20Loss_moreDA_3_5kep_EDT25__nnUNetResEncUNetMPlans__3d_fullres_bs16_ps128_256_256",
    "--fold", "('all', )", 
    "--threshold", "0.15",
]

p0 = subprocess.Popen(base + ["--gpu-id", "0", "--output-file", "/kaggle/working/submission_gpu0.csv"])
p1 = subprocess.Popen(base + ["--gpu-id", "1", "--output-file", "/kaggle/working/submission_gpu1.csv"])
p0.wait(); p1.wait()

nnUNet_raw is not defined and nnU-Net can only be used on data for which preprocessed files are already present on your system. nnU-Net cannot be used for experiment planning and preprocessing like this. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up properly.
nnUNet_preprocessed is not defined and nnU-Net can not be used for preprocessing or training. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up.
nnUNet_results is not defined and nnU-Net cannot be used for training or inference. If this is not intended behavior, please read documentation/setting_up_paths.md for information on how to set this up.
Using torch.compile


  0%|          | 0/18 [00:00<?, ?it/s]W0617 17:11:27.233000 45 torch/_inductor/utils.py:1250] [0/0] Not enough SMs to use max_autotune_gemm mode


nnUNet_raw is not defined and nnU-Net can only be used on data for which preprocessed files are already present on your system. nnU-Net cannot be used for experiment planning and preprocessing like this. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up properly.
nnUNet_preprocessed is not defined and nnU-Net can not be used for preprocessing or training. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up.
nnUNet_results is not defined and nnU-Net cannot be used for training or inference. If this is not intended behavior, please read documentation/setting_up_paths.md for information on how to set this up.
Using torch.compile


100%|██████████| 18/18 [02:33<00:00,  8.55s/it]


Saved predictions to /kaggle/working/submission_gpu1.csv


100%|██████████| 18/18 [01:06<00:00,  3.72s/it]


Saved predictions to /kaggle/working/submission_gpu0.csv


0

## Merge resulting CSVs

In [5]:
# write header from first file, then append data (skipping headers) from both
!(head -n 1 submission_gpu0.csv && tail -n +2 -q submission_gpu0.csv submission_gpu1.csv) > submission.csv

In [6]:
# print the final prediction file
!head submission.csv

tomo_id,Motor axis 0,Motor axis 1,Motor axis 2
tomo_003acc,-1,-1,-1
tomo_01a877,145,640,285
tomo_00e047,171,544,600
